# Tutorial: Volume 1 Chapter 7 Counterfactual Analysis

Source: `Volume 1 Chapter 7`

This tutorial implements the **Zero-Torque Counterfactual (ZTCF)** framework for dynamical biomechanics.
Given a factual trajectory driven by muscle torques $u(t)$, the counterfactual trajectory $\tilde{x}(t)$ branches at intervention time $t^*$ by setting all control torques to zero:
$$u_{\text{counterfactual}}(t) = 0, \quad \forall t \ge t^*$$

By integrating forward from the identical state $x(t^*)$, we isolate the exact causal contribution of active control versus passive structural dynamics.

In [ ]:
from __future__ import annotations

import numpy as np

from src.affine_control.dynamics import (
    christoffel_coriolis,
    double_pendulum_coriolis,
    double_pendulum_mass_matrix,
    planar_double_pendulum_trajectory,
)

CHAPTER = "Volume 1 Chapter 7"
print(f"Loaded tutorial for {CHAPTER}")

## 1. Planar Double Pendulum Model & Potential Energy

We model the torso/arm and club as a 2-segment planar linkage with point-mass rods.
The gravity potential gradient $\nabla V(q)$ provides the conservative drift forces.

In [ ]:
m1, m2 = 2.0, 0.35  # link masses (kg)
l1, l2 = 0.60, 0.90  # link lengths (m)
g_acc = 9.81  # m/s^2


def mass_matrix(q: np.ndarray) -> np.ndarray:
    return double_pendulum_mass_matrix(q, m1=m1, m2=m2, l1=l1, l2=l2)


def grad_potential(q: np.ndarray) -> np.ndarray:
    q1, q2 = float(q[0]), float(q[1])
    # dV/dq for planar double pendulum in vertical plane
    dV_dq1 = (m1 / 2.0 + m2) * g_acc * l1 * np.sin(q1)
    dV_dq2 = (m2 / 2.0) * g_acc * l2 * np.sin(q2)
    return np.array([dV_dq1, dV_dq2])


# Initial condition at start of downswing: arms raised, wrist cocked
q0 = np.array([np.pi * 0.75, np.pi * 0.25])
qd0 = np.array([1.5, 0.5])  # rad/s

print("Initial Configuration q0:", q0)
print("Initial Velocity qd0:", qd0)

## 2. Energy Conservation & Skew-Symmetry Verification

The Christoffel Coriolis matrix construction ensures that:
$$\dot{M}(q) - 2 C(q, \dot{q})$$
is skew-symmetric, which is the necessary and sufficient condition for power conservation $\frac{d}{dt}\left(\frac{1}{2}\dot{q}^T M \dot{q}\right) = \dot{q}^T \tau$.

In [ ]:
C = christoffel_coriolis(mass_matrix, q0, qd0)
C_analytic = double_pendulum_coriolis(q0, qd0, m2=m2, l1=l1, l2=l2)

# Compute M_dot via chain rule dM/dt = sum (dM/dq_k * qd_k)
eps = 1e-6
M_plus = mass_matrix(q0 + eps * qd0)
M_minus = mass_matrix(q0 - eps * qd0)
M_dot = (M_plus - M_minus) / (2.0 * eps)

skew_test = M_dot - 2.0 * C
symmetric_part = 0.5 * (skew_test + skew_test.T)
skew_norm = np.linalg.norm(symmetric_part)

print("Coriolis matrix C(q, qd):\n", C)
print(f"Skew-symmetry violation norm: {skew_norm:.2e}")
assert skew_norm < 1e-5, "Mdot - 2C must be skew-symmetric to preserve passive energy!"

## 3. Forward Simulation: Active Baseline vs. Zero-Torque Counterfactual

We simulate two branches over a 0.20-second downswing horizon:
1. **Active Baseline:** Constant driving torques $\tau_{\text{active}} = [25.0, 5.0]\text{ Nm}$.
2. **ZTCF Branch:** Torques dropped to zero $\tau_{\text{counterfactual}} = [0.0, 0.0]\text{ Nm}$.

In [ ]:
horizon = 0.20  # seconds
steps = 200

# 1. Baseline simulation
traj_baseline = planar_double_pendulum_trajectory(
    mass_matrix=mass_matrix,
    grad_potential=grad_potential,
    q0=q0,
    qd0=qd0,
    torque=np.array([25.0, 5.0]),
    horizon=horizon,
    steps=steps,
)

# 2. Counterfactual zero-torque simulation
traj_ztcf = planar_double_pendulum_trajectory(
    mass_matrix=mass_matrix,
    grad_potential=grad_potential,
    q0=q0,
    qd0=qd0,
    torque=np.array([0.0, 0.0]),
    horizon=horizon,
    steps=steps,
)

print(f"Simulated {len(traj_baseline)} time steps for both branches.")

## 4. Clubhead Speed & Dynamics Attribution

We extract terminal velocities and calculate distal tip speed $v_{\text{clubhead}}$:
$$v_{\text{clubhead}} = \|\dot{p}_{\text{tip}}\|$$
comparing the active baseline against the passive dynamics counterfactual.

In [ ]:
# Compute clubhead speed from generalized coordinates (q, qd)
def clubhead_speed(q: np.ndarray, qd: np.ndarray) -> float:
    q1, q2 = float(q[0]), float(q[1])
    d1, d2 = float(qd[0]), float(qd[1])
    vx = -l1 * np.sin(q1) * d1 - l2 * np.sin(q2) * d2
    vy = l1 * np.cos(q1) * d1 + l2 * np.cos(q2) * d2
    return float(np.hypot(vx, vy))


# Initial speed
v_init = clubhead_speed(traj_baseline[0][1], traj_baseline[0][2])

# Terminal speeds at t = 200 ms
t_end_base, q_end_base, qd_end_base = traj_baseline[-1]
t_end_ztcf, q_end_ztcf, qd_end_ztcf = traj_ztcf[-1]

v_base = clubhead_speed(q_end_base, qd_end_base)
v_ztcf = clubhead_speed(q_end_ztcf, qd_end_ztcf)

delta_speed = v_base - v_ztcf
passive_percentage = (v_ztcf / v_base) * 100.0

print(f"Initial Speed:      {v_init:.2f} m/s")
print(f"Baseline Terminal:  {v_base:.2f} m/s")
print(f"ZTCF Terminal:      {v_ztcf:.2f} m/s")
print(f"Active Torque Gain: {delta_speed:.2f} m/s")
print(f"Passive Dynamics Fraction: {passive_percentage:.1f}%")

assert v_base > v_ztcf, "Active torque must produce higher clubhead speed than ZTCF!"
assert v_ztcf > 0.0, "Passive momentum must produce non-zero terminal speed!"

## 5. Verification Summary

The Zero-Torque Counterfactual demonstrates:
- Passive dynamics accounts for a substantial fraction of delivered velocity.
- The Christoffel Coriolis implementation eliminates non-physical energy drift.
- Causal attribution separates torque contribution from inertial propagation.

In [ ]:
summary = {
    "chapter": CHAPTER,
    "ztcf_divergence_computed": True,
    "baseline_terminal_speed_mps": round(v_base, 2),
    "ztcf_terminal_speed_mps": round(v_ztcf, 2),
    "passive_fraction_percent": round(passive_percentage, 1),
}
print("Verification Summary:", summary)